# A Deep Agent is just a regular agent with
- A small collection of essential tools
- A small collection of prebuilt middleware

### a few superpowers
- The ability to read from and write to filesystems
- The ability to spawn (sync/async) subagents

### and the ability to plan
- The builtin **todo list** middleware

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from util import pretty_print, print_todos, print_exchange, print_activity
from langchain_core.tools import tool
from tavily import TavilyClient
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI
import os
from langchain.agents.middleware import TodoListMiddleware

from dotenv import load_dotenv
load_dotenv(override=True)

# model = "claude-haiku-4-5-20251001"

model = ChatOpenAI(
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    # model_name="gpt-5.5",
    model_name="gpt-5.6-luna",
    # Deep Agents uses function tools; reasoning + tools requires /v1/responses.
    use_responses_api=True,
    streaming=False,
    
)
    
research_system_prompt = """You are an expert research assistant."""

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    # middleware=[TodoListMiddleware()]
)

result = agent.invoke({"messages": [{"role": "user", "content": "What tools do you have?"}]})
pretty_print(result)

### Human

What tools do you have?

### Ai

I can:

- **Search and inspect files**: list directories, find files by pattern, search for literal text, and read file contents.
- **Edit files**: make precise replacements, create/overwrite files, and delete files or directories.
- **Delegate complex tasks**: launch a research subagent to investigate files or perform multi-step analysis.
- **Work with images and PDFs**: read their available content, including multimodal data when supported.
- **Answer questions and help research or write** based on information you provide or that’s available in the workspace.

I don’t have a general web-browsing tool in this environment.

# The TODO list in action

#### Ask a deep agent to perform a complex task and it starts by planning out all the steps.

In [3]:
result = agent.invoke({"messages": [{"role": "user", "content": "I need to bake a cake. Make a plan. Use TodoList Middleware"}]})
print_todos(result)

*No todos.*

# Tools

#### Deep agents, as well as their subagents, can be extended with tools. Let's make ours more useful by allowing it to search the internet.

In [4]:
# @tool
# def search(query: str) -> str:
#     """Search the internet for information

#     Args:
#         query (str): search query
#     """
#     results = TavilyClient().search(query)
#     return results.get("results", [])

tools=[{"type": "web_search"}]

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    # tools=[search]
    tools=[{"type": "web_search"}]
)

result = agent.invoke({"messages": [{"role": "user", "content": "What are the top 2 stories in the news today?"}]})
pretty_print(result)

### Human

What are the top 2 stories in the news today?

### Ai

I don’t have access to live news feeds in this chat, so I can’t reliably identify today’s top two stories. If you share a news homepage or a few headlines—and specify your country or preferred outlet—I can rank and summarize them.

# Task Delegation: Where Deep Agents Shine

#### The typical pattern is one main supervisor agent delegating to one or more specialized subagents.

### Each subagent has:
- its own context window
- its own tools
- its own system prompt
- its own model (can override the main agent's model)
- its own skills

### The main agent
- delegates via its task() tool
- sees only the final results of its subagents, keeping its context window clean

![Deep agent task delegation](images/deep-agent-delegation.svg)

# Defining Subagents: Dictionary Approach

#### Just define your subagent as a Python **dictionary** and plug it into your supervisor main agent.

In [28]:
research_subagent = {
    "name": "research-agent",
    "description": "Research a given task",
    "system_prompt": f"""You are a research assistant.
                         Use tools to gather information.
                         Structure findings with clear headings and inline citations.
                         Limit to 3 search calls.""",
    "model": "anthropic:claude-haiku-4-5-20251001",
    "tools": [search]
}

agent = create_deep_agent(model=model, system_prompt=research_system_prompt, subagents=[research_subagent])

# Defining Subagents: Compiled Approach

For more complex workflows we can use **compiled** subagents, allowing us arbitrary complexity.

In [30]:
from langchain.agents import create_agent
from deepagents import CompiledSubAgent

research_subagent = create_agent(
    model=model,
    tools=[search],
    system_prompt="""You are a research assistant.
                     Use the search tool to gather information.
                     Structure findings with clear headings and inline citations.
                     Limit to 3 search calls.""",
)

compiled_subagent = CompiledSubAgent(
    name="research-agent", description="Research a given task using web search", runnable=research_subagent,
)

agent = create_deep_agent(model=model, system_prompt=research_system_prompt, subagents=[compiled_subagent])

# Filesystems and backends

#### Deep agents are designed to handle long-running, complex tasks. They typically do this by both offloading excess context that accumulates during execution and decomposing tasks into sub-tasks and delegating execution to subagents. Deep agents expose a virtual **filesystem** surface to allow (sub)agents to read and write state through a collection of tools: `ls, read_file, write_file, edit_file, glob, and grep`. The actual backend type is pluggable and abstracted from the agent.

A few prebuilt filesystem backends that you can quickly use with your deep agent:

<table style="font-size:18px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:12px; text-align:left;">Backend</th>
<th style="border:1px solid #999; padding:12px; text-align:left;">Description</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Default<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash")</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Thread-scoped. The default filesystem backend for an agent is stored in langgraph state. Files persist across turns within a thread (via your checkpointer) and are not shared across threads.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Local filesystem persistence<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=FilesystemBackend(root_dir="/Users/nh/Desktop/"))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Gives the deep agent access to your local machine's filesystem. You can specify the root directory that the agent has access to. Any provided <code style="font-size:15px; color:#2b8a3e;">root_dir</code> must be an absolute path. Typically, wrap in a <code style="font-size:15px; color:#2b8a3e;">CompositeBackend</code> to keep internal agent data (offloaded tool results, conversation history) separate from your project files.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Durable store (LangGraph store)<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=StoreBackend())</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Gives the agent access to long-term storage that is persisted across threads. Great for storing longer term memories or instructions that are applicable to the agent over multiple executions.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Context Hub<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=ContextHubBackend("my-agent"))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Stores files durably in a LangSmith Hub repo, without provisioning a separate LangGraph store.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Sandbox<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=sandbox)</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Execute code in isolated environments. Sandboxes provide filesystem tools plus the <code style="font-size:15px; color:#2b8a3e;">execute</code> tool for running shell commands. Choose from LangSmith, AgentCore, Daytona, Deno, E2B, Modal, Runloop, or local VFS.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Local shell<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=LocalShellBackend(root_dir=".", env={"PATH": "/usr/bin:/bin"}))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Filesystem and shell execution directly on the host. No isolation—use only in controlled development environments. See security considerations below.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Composite</td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Thread-scoped by default, <code style="font-size:15px; color:#2b8a3e;">/memories/</code> persisted across threads. The Composite backend is maximally flexible. You can specify different routes in the filesystem to point towards different backends.</td>
</tr>
</tbody>
</table>

# Default Backend
- Default backend is thread-scoped
- Without a `thread_id` the agent cannot find the referenced filesystem
- Stored in-memory, or SQLLite, Postgres, Redis, MongoDB

####  Thread ID is how we tell our agent that our response is part of a larger conversation.

In [31]:
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model=model, system_prompt=research_system_prompt, checkpointer=MemorySaver(),  # We don't have to mention the backend when using the 'default'; it is just available to us
)

config = {"configurable": {"thread_id": "joke-session"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Tell me a joke about computer scientists and write it to /joke.md"}]
}, config=config)
pretty_print(result)

### Human

Tell me a joke about computer scientists and write it to /joke.md

### Tool

Updated file /joke.md

### Ai

Wrote the joke to `/joke.md`.

#### We are able to read the filesystem because we pass in the same `config` object has the `thread_id` we previously used.

In [32]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /joke.md and tell me the joke"}]
}, config=config)
print_exchange(result)

### Human

Tell me a joke about computer scientists and write it to /joke.md

### Ai

Why do computer scientists confuse Halloween and Christmas?

Because OCT 31 == DEC 25.

#### This time the agent cannot find the file because we use a different `thread_id`

In [33]:
other_config = {"configurable": {"thread_id": "other-session"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /joke.md and tell me the joke"}]
}, config=other_config)
print_exchange(result)

### Human

Read /joke.md and tell me the joke

### Ai

I couldn’t read `/joke.md` because the file wasn’t found at that path.

# Store Backend
- Long-term
- Backed by a database
- Persisted across threads
- Good for storing longer term memories or instructions that are applicable to the agent over multiple executions.

#### In this example we write a memory in a thread named 'session-a'

In [36]:
from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    backend=StoreBackend(namespace=lambda ctx: ("memories",)),
    store=store,
)

# Write a memory in one thread...
session_a_config = {"configurable": {"thread_id": "session-a"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Remember that my name is Faisal by writing it to /profile.md"}]
}, config=session_a_config)

pretty_print(result)

### Human

Remember that my name is Faisal by writing it to /profile.md

### Tool

Updated file /profile.md

### Ai

Done.

#### Now we can read the memory back from a _different_ thread. Unlike the default backend, the **StoreBackend** persists files across threads.

In [37]:
session_b_config = {"configurable": {"thread_id": "session-b"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /profile.md and tell me my name"}]
}, config=session_b_config)

print_exchange(result)

### Human

Read /profile.md and tell me my name

### Ai

Your name is Faisal.

In [39]:
# Per-user long-term memory: shared across a user's threads, isolated from every other user.
from dataclasses import dataclass
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

@dataclass(frozen=True)
class UserContext:
    user_id: str  # In production, populate this from authenticated server-side identity.

def user_memory_namespace(runtime):
    if runtime.context is None or not runtime.context.user_id:
        raise PermissionError("Authenticated user required")
    return ("deepagents-basic", "v1", "users", runtime.context.user_id, "memories")

# InMemoryStore keeps this demo self-contained; use a database-backed BaseStore in production.
user_store = InMemoryStore()
user_memory_backend = CompositeBackend(
    default=StateBackend(),  # Scratch files remain scoped to each thread.
    routes={"/memories/": StoreBackend(namespace=user_memory_namespace)},
)

user_scoped_agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    backend=user_memory_backend,
    store=user_store,
    context_schema=UserContext,
    checkpointer=MemorySaver(),
)

def invoke_as(user_id: str, thread_id: str, message: str):
    return user_scoped_agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        config={"configurable": {"thread_id": thread_id}},
        context=UserContext(user_id=user_id),
    )

# User A writes in one thread.
invoke_as("user-a", "user-a-thread-1", "Remember that my name is Faisal by writing it to /memories/profile.md")

# User A can read the same file in a new thread.
same_user = invoke_as(
    "user-a", "user-a-thread-2",
    "Read /memories/profile.md. If it exists, tell me the name; otherwise say 'No memory found'.",
)
print("Same user, different thread:")
print_exchange(same_user)

# User B asks for the identical path but receives an isolated namespace.
different_user = invoke_as(
    "user-b", "user-b-thread-1",
    "Read /memories/profile.md. If it exists, tell me the name; otherwise say 'No memory found'.",
)
print("\nDifferent user:")
print_exchange(different_user)


Same user, different thread:


### Human

Read /memories/profile.md. If it exists, tell me the name; otherwise say 'No memory found'.

### Ai

Faisal


Different user:


### Human

Read /memories/profile.md. If it exists, tell me the name; otherwise say 'No memory found'.

### Ai

No memory found

# Store Backend: Choosing the backing database

The <code style="font-size:15px; color:#2b8a3e;">StoreBackend</code> is <b>storage-agnostic</b>. It is a thin filesystem adapter over whatever LangGraph <code style="font-size:15px; color:#2b8a3e;">BaseStore</code> you pass to <code style="font-size:15px; color:#2b8a3e;">create_deep_agent(store=...)</code>. The database is chosen entirely by that one argument &mdash; the agent code never changes.

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code>agent = create_deep_agent(
    model=model,
    backend=StoreBackend(namespace=lambda ctx: ("memories",)),
    store=<span style="color:#cf222e;">&lt;ANY BaseStore&gt;</span>,   <span style="color:#6e7781;"># &larr; this is the database choice</span>
)</code></pre>

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;">Store</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Package</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Import</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">InMemoryStore</code> <span style="color:#6e7781;">(dev only)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left;">built-in</td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.memory</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">PostgresStore</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph-checkpoint-postgres</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.postgres</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">RedisStore</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph-checkpoint-redis</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.redis</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">MongoDBStore</code></b></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">langgraph-store-mongodb</code></b></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">langgraph.store.mongodb</code></b></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;">Anything custom</td>
<td style="border:1px solid #999; padding:10px; text-align:left;">your code</td>
<td style="border:1px solid #999; padding:10px; text-align:left;">subclass <code style="color:#2b8a3e;">BaseStore</code></td>
</tr>
</tbody>
</table>

### MongoDB example

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code><span style="color:#cf222e;">from</span> langgraph.store.mongodb <span style="color:#cf222e;">import</span> MongoDBStore

<span style="color:#cf222e;">with</span> MongoDBStore.from_conn_string(
    conn_string=<span style="color:#0a3069;">"mongodb://localhost:27017"</span>,
    db_name=<span style="color:#0a3069;">"deepagents"</span>,
    collection_name=<span style="color:#0a3069;">"memories"</span>,
) <span style="color:#cf222e;">as</span> store:
    agent = create_deep_agent(
        model=model,
        system_prompt=research_system_prompt,
        backend=StoreBackend(namespace=<span style="color:#cf222e;">lambda</span> ctx: (<span style="color:#0a3069;">"memories"</span>,)),
        store=store,   <span style="color:#6e7781;"># only this line changed &mdash; files now persist in MongoDB</span>
    )
    <span style="color:#6e7781;"># ...same invoke() calls as the InMemoryStore cell</span></code></pre>


# Composite Backend
- Flexible: specify different routes in the filesystem to point towards different backends.

#### Example:
- Use <code style="font-size:15px; color:#2b8a3e;">StateBackend</code> for thread-scoped default storage
- Save persistent memories to <code style="font-size:15px; color:#2b8a3e;">/memories/</code>. These files are available from different threads.
- Save ephemeral memories to the root <code style="font-size:15px; color:#2b8a3e;">/</code>. These are not available across different threads.



In [40]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# Route by path: everything is ephemeral (StateBackend) except /memories/, which is routed to the durable StoreBackend.
composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=lambda ctx: ("memories",))},
)

agent = create_deep_agent(
    model=model, system_prompt=research_system_prompt, backend=composite_backend, store=store,
)

# Thread A: write one ephemeral file and one durable file
result = agent.invoke({
    "messages": [{"role": "user", "content": (
        "Write a quick scratch note to /draft.md, and save that my name is Faisal to /memories/profile.md"
    )}]
}, config={"configurable": {"thread_id": "thread-a"}})
pretty_print(result)

### Human

Write a quick scratch note to /draft.md, and save that my name is Andrew to /memories/profile.md

### Tool

['/memories/']

### Tool

Updated file /draft.md

### Tool

Updated file /memories/profile.md

### Ai

Done.

In [41]:
# Thread B: a *different* thread. The /memories/ route persisted; /draft.md did not.
result = agent.invoke({
    "messages": [{"role": "user", "content": (
        "Try to read both /memories/profile.md and /draft.md. Report exactly what you find for each."
    )}]
}, config={"configurable": {"thread_id": "thread-b"}})
print_exchange(result)

### Human

Try to read both /memories/profile.md and /draft.md. Report exactly what you find for each.

### Ai

`/memories/profile.md`:

```md
# Profile

Name: Andrew
```

`/draft.md`:

File not found.

# Context Isolation vs. Shared Filesystems

When a supervisor delegates to a subagent, two different things happen to two different kinds of state:

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left; width:22%;">What</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Behavior</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Message context</b><br><span style="color:#6e7781;">(isolated)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The subagent runs in its <b>own context window</b>. Its intermediate reasoning and tool calls never enter the supervisor's history &mdash; only a single, compact <b>final report</b> is returned. This is the whole point of subagents: keep the supervisor's context clean.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Filesystem</b><br><span style="color:#6e7781;">(shared)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The backend is <b>shared</b> between the supervisor and all subagents. Any file a subagent writes <b>remains in state after the subagent finishes</b> and stays readable by the supervisor and sibling subagents.</td>
</tr>
</tbody>
</table>

### The pattern this unlocks

The filesystem is a **shared side-channel** for moving large results across the subagent boundary *without* paying for them in context:

1. Supervisor delegates: *"research X and write the findings to <code style="color:#2b8a3e;">/research.md</code>"*
2. Subagent does the heavy work, writes the file, and returns just *"wrote results to /research.md"*
3. Supervisor (or another subagent) reads <code style="color:#2b8a3e;">/research.md</code> when it needs the detail

The bulky output travels through the filesystem; only a one-line handoff travels through the conversation.

<b>Caveat:</b> dictionary subagents inherit the supervisor's backend automatically. A fully custom <code style="font-size:15px; color:#2b8a3e;">CompiledSubAgent</code> graph has its own state and only shares the filesystem if you wire it to the same backend.

In [25]:
from langgraph.checkpoint.memory import MemorySaver

# A subagent whose job is to write its findings to a file (not to report them inline).
writer_subagent = {
    "name": "writer-agent",
    "description": "Writes the requested content to a file, then reports only the path.",
    "system_prompt": (
        "Write the requested content to the requested file path using write_file. "
        "Then report ONLY the path you wrote to — do not repeat the content."
    ),
    "tools": [],
}

agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a supervisor. Delegate writing tasks to the writer-agent subagent via the task tool. "
        "The subagent shares your filesystem, so after it finishes you can read the file it wrote yourself."
    ),
    subagents=[writer_subagent],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "subagent-fs-demo"}}

# print_activity streams the run with subgraphs=True, so we can watch the *subagent*
# call write_file inside its own isolated context — that tool call never appears in
# the supervisor's returned messages.
print_activity(agent, (
    "Delegate to writer-agent: have it write a haiku about the ocean to /poem.md. "
    "After the subagent finishes, YOU read /poem.md with read_file and quote its exact contents back to me."
), config=config)

### Activity timeline

- 🧑‍✈️ **supervisor** → 📨 delegates via **task** to `writer-agent`
    - 🤖 **writer-agent** → 🔧 **write_file**(`/poem.md`, content=“Tides breathe under moonlight Salt wind combs the sleeping waves Blue heart hums…”)
    - 🤖 **writer-agent** ← 📥 `write_file` → Updated file /poem.md
- 🧑‍✈️ **supervisor** ← 📥 `task` → /poem.md
- 🧑‍✈️ **supervisor** → 🔧 **read_file**(`/poem.md`)
- 🧑‍✈️ **supervisor** ← 📥 `read_file` → 1 Tides breathe under moonlight 2 Salt wind combs the sleeping waves 3 Blue hear…

#### Because we passed a checkpointer + thread_id, the run's state is durable on this thread. The subagent's file lives in the shared filesystem, readable by the main supervisor agent.

In [26]:
state = agent.get_state(config)
print("Files in shared state:", list(state.values.get("files", {}).keys()))

# And the supervisor's own transcript only ever saw the delegation + the final read —
# never the subagent's write_file call (that stayed isolated in the subagent's context).
pretty_print(state.values)

Files in shared state: ['/poem.md']


### Human

Delegate to writer-agent: have it write a haiku about the ocean to /poem.md. After the subagent finishes, YOU read /poem.md with read_file and quote its exact contents back to me.

### Tool

/poem.md

### Tool

1  Tides breathe under moonlight
2  Salt wind combs the sleeping waves
3  Blue heart hums softly

### Ai

Here are the exact contents of `/poem.md`:

```text
Tides breathe under moonlight
Salt wind combs the sleeping waves
Blue heart hums softly
```

# Context Management Techniques

### A. Subagent Context Isolation

Each subagent gets its own, isolated context window. This ensures that context relevant to a specific task is only seen by the subagent working on that task.

### B. Offloading Large Inputs and Outputs

When a tool returns (or receives) more than comfortably fits in context, the harness writes the payload to the filesystem and replaces it in the message history with a short **reference plus a preview** (e.g. the first 10 lines). The agent calls <code style="color:#2b8a3e;">read_file</code> to pull back more **only if it actually needs the detail** — instead of dragging thousands of lines through every subsequent turn.

<img src="images/offloading-results.png" alt="Offloading large tool results to the filesystem" style="max-width:100%; width:820px; border:1px solid #d0d7de; border-radius:6px;" />

<ul style="font-size:16px;">
<li>Oversized tool results are evicted and swapped for a reference.</li>
<li>The agent reads back just the slice it needs via <code style="color:#2b8a3e;">read_file</code>.</li>
</ul>

### C. Context Summarization

When the **conversation itself** approaches the model's limit (~85% of <code style="color:#2b8a3e;">max_input_tokens</code>), <code style="color:#2b8a3e;">SummarizationMiddleware</code> runs a **compaction step**: older turns are compressed into an LLM-generated summary — session intent, artifacts created, next steps — while the most recent ~10% of tokens are kept verbatim.

<ul style="font-size:16px;">
<li><b>In-context summary + pointer:</b> the summary replaces the old messages and tells the agent where the full record lives.</li>
<li><b>Filesystem preservation:</b> the original messages are written to <code style="color:#2b8a3e;">/conversation_history/{thread_id}.md</code>, so the agent can search or re-read them if needed.</li>
</ul>

<img src="images/summarization.png" alt="Summarizing older conversation turns into a compact summary + pointer" style="max-width:100%; width:820px; border:1px solid #d0d7de; border-radius:6px;" />

